# 01 - Rawasi Data Understanding and EDA

Purpose: understand the Rawasi synthetic invoice discounting dataset before building filters, benchmark features, anomaly checks, or credit models.

This notebook focuses on:

- file shape, schema, missingness, and data quality
- target distribution and out-of-time split health
- leakage review
- numeric and categorical distributions
- segment-level default patterns
- candidate hard-filter diagnostics
- first-pass peer-group outlier analysis

No model training happens here.

## 1. Setup

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Seaborn imports scipy, and some local Python installs can have a numpy/scipy
# binary mismatch. Use seaborn when it works; otherwise fall back to matplotlib.
try:
    import seaborn as sns
except Exception as exc:
    print(f'Seaborn unavailable, using matplotlib fallback: {exc}')

    class _SeabornFallback:
        def set_theme(self, style='whitegrid', context='notebook'):
            try:
                plt.style.use('seaborn-v0_8-whitegrid')
            except OSError:
                plt.style.use('default')

        def _colors(self, color=None, palette=None, n=1):
            if color is not None:
                return [color] * n
            if palette == 'Set2':
                return list(plt.cm.Set2(np.linspace(0, 1, max(n, 1))))
            if palette == 'coolwarm':
                return list(plt.cm.coolwarm(np.linspace(0, 1, max(n, 1))))
            return [plt.rcParams['axes.prop_cycle'].by_key()['color'][0]] * n

        def barplot(self, data, x, y, ax=None, color=None, palette=None, **kwargs):
            ax = ax or plt.gca()
            plot_data = data[[x, y]].copy()
            x_is_num = pd.api.types.is_numeric_dtype(plot_data[x])
            y_is_num = pd.api.types.is_numeric_dtype(plot_data[y])
            n = len(plot_data)
            colors = self._colors(color=color, palette=palette, n=n)

            if y_is_num and not x_is_num:
                labels = plot_data[x].astype(str).tolist()
                values = plot_data[y].to_numpy()
                ax.bar(range(n), values, color=colors)
                ax.set_xticks(range(n))
                ax.set_xticklabels(labels)
            else:
                labels = plot_data[y].astype(str).tolist()
                values = plot_data[x].to_numpy()
                ax.barh(range(n), values, color=colors)
                ax.set_yticks(range(n))
                ax.set_yticklabels(labels)
            return ax

        def histplot(self, data, kde=False, bins=35, ax=None, color=None, **kwargs):
            ax = ax or plt.gca()
            values = pd.Series(data).dropna()
            ax.hist(values, bins=bins, color=color or '#4c78a8', alpha=0.85)
            return ax

        def boxplot(self, data, x, y, ax=None, showfliers=True, palette=None, **kwargs):
            ax = ax or plt.gca()
            groups = list(pd.Series(data[x]).dropna().unique())
            values = [data.loc[data[x].eq(g), y].dropna().to_numpy() for g in groups]
            bp = ax.boxplot(values, labels=[str(g) for g in groups], showfliers=showfliers, patch_artist=True)
            colors = self._colors(palette=palette, n=len(groups))
            for patch, c in zip(bp['boxes'], colors):
                patch.set_facecolor(c)
            return ax

    sns = _SeabornFallback()

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (11, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11

In [ ]:
# Resolve the project root whether this notebook is run from the repo root or from notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'utils').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_CANDIDATES = sorted((PROJECT_ROOT / 'utils').glob('*synthetic*.csv*'))
if not DATA_CANDIDATES:
    DATA_CANDIDATES = sorted((PROJECT_ROOT / 'utils').glob('*.csv*'))
if len(DATA_CANDIDATES) != 1:
    raise FileNotFoundError(f'Expected one dataset CSV under utils/, found: {DATA_CANDIDATES}')
DATA_PATH = DATA_CANDIDATES[0]
DATE_COLS = ['invoice_date', 'expected_payment_date', 'actual_resolution_date']

df = pd.read_csv(DATA_PATH, parse_dates=DATE_COLS)

print(f'Dataset: {DATA_PATH}')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
df.head()

## 1A. Run Stage A and Stage B Scripts

These cells execute the project scripts and load their CSV outputs back into dataframes. Use them when you want to inspect the actual pipeline outputs rather than recomputing logic manually inside the notebook.

In [ ]:
import subprocess
import sys

def run_project_script(script_name):
    script_path = PROJECT_ROOT / 'scripts' / script_name
    result = subprocess.run(
        [sys.executable, str(script_path)],
        cwd=PROJECT_ROOT,
        check=True,
        text=True,
        capture_output=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result


### Stage A Script Output

In [ ]:
run_project_script('run_stage_a_filters.py')

stage_a_df = pd.read_csv(
    PROJECT_ROOT / 'reports' / 'stage_a_scored_transactions.csv',
    parse_dates=DATE_COLS,
)
stage_a_rule_summary = pd.read_csv(PROJECT_ROOT / 'reports' / 'stage_a_rule_summary.csv')

display(stage_a_df.head())
display(stage_a_df['stage_a_decision'].value_counts().to_frame('rows'))
display(stage_a_rule_summary)

### Stage B Script Output

In [ ]:
run_project_script('run_stage_b_benchmarks.py')

stage_b_df = pd.read_csv(
    PROJECT_ROOT / 'reports' / 'stage_b_scored_transactions.csv',
    parse_dates=DATE_COLS,
)
stage_b_feature_summary = pd.read_csv(PROJECT_ROOT / 'reports' / 'stage_b_feature_summary.csv')
stage_b_peer_outliers = pd.read_csv(PROJECT_ROOT / 'reports' / 'stage_b_peer_outliers.csv')

stage_b_columns_to_view = [
    'transaction_id', 'stage_b_split', 'defaulted',
    'ticket_to_revenue_est', 'outstanding_to_revenue_est', 'advance_tenor_exposure',
    'stage_b_benchmark_score', 'stage_b_benchmark_band',
    'stage_b_peer_outlier_count', 'stage_b_peer_outlier_reasons',
]

display(stage_b_df[stage_b_columns_to_view].head())
display(pd.crosstab(stage_b_df['stage_b_split'], stage_b_df['stage_b_benchmark_band']))
display(stage_b_feature_summary)
display(stage_b_peer_outliers.head(20))

## 2. Schema and Data Quality

In [ ]:
schema = pd.DataFrame({
    'column': df.columns,
    'dtype': [df[c].dtype for c in df.columns],
    'non_null': [df[c].notna().sum() for c in df.columns],
    'nulls': [df[c].isna().sum() for c in df.columns],
    'unique_values': [df[c].nunique(dropna=False) for c in df.columns],
})
schema

In [ ]:
print('Duplicate transaction IDs:', df['transaction_id'].duplicated().sum())
print('Unique exporters:', df['exporter_id'].nunique())
print('Unique buyers:', df['buyer_id'].nunique())
print('Invoice date range:', df['invoice_date'].min().date(), 'to', df['invoice_date'].max().date())

missing = (
    df.isna().sum()
    .rename('missing_count')
    .to_frame()
    .assign(missing_pct=lambda x: x['missing_count'] / len(df))
    .query('missing_count > 0')
    .sort_values('missing_count', ascending=False)
)
missing

In [ ]:
bool_cols = [c for c in df.columns if df[c].dropna().isin([True, False]).all() and df[c].dtype == bool]
object_cols = df.select_dtypes(include='object').columns.tolist()
numeric_cols = df.select_dtypes(include='number').columns.tolist()

print('Boolean columns:', bool_cols)
print('\nObject columns and cardinality:')
display(df[object_cols].nunique(dropna=False).sort_values(ascending=False).to_frame('n_unique'))

print('\nNumeric columns:', numeric_cols)

## 3. Leakage Review

`invoice_date` is the decision date. Any value that depends on what happened after `invoice_date` must be excluded from model features.

In [ ]:
LEAKY_OR_OUTCOME_COLS = [
    'defaulted',
    'status',
    'actual_resolution_date',
    'days_to_resolution',
    'buy_buyer_baseline_default_prob',
]

REDUNDANT_DECISION_TIME_COLS = [
    'expected_payment_date',  # Known at decision time, but exactly invoice_date + payment_terms_days.
]

ID_COLS = ['transaction_id', 'exporter_id', 'buyer_id']

feature_exclusions = LEAKY_OR_OUTCOME_COLS + REDUNDANT_DECISION_TIME_COLS + ID_COLS
candidate_feature_cols = [c for c in df.columns if c not in feature_exclusions]

print('Leakage/outcome columns present:')
print([c for c in LEAKY_OR_OUTCOME_COLS if c in df.columns])
print('\nRedundant decision-time columns present:')
print([c for c in REDUNDANT_DECISION_TIME_COLS if c in df.columns])
print(f'\nCandidate feature columns after exclusions: {len(candidate_feature_cols)}')
candidate_feature_cols

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

resolved = df[df['defaulted'].notna()].copy()
resolved['defaulted'] = resolved['defaulted'].astype(int)

leakage_checks = []
for col in ['days_to_resolution', 'buy_buyer_baseline_default_prob', 'buy_buyer_country_risk_score', 'exp_iscore']:
    if col in resolved.columns and resolved[col].notna().all():
        leakage_checks.append({
            'column': col,
            'roc_auc_single_feature': roc_auc_score(resolved['defaulted'], resolved[col]),
            'auc_pr_single_feature': average_precision_score(resolved['defaulted'], resolved[col]),
        })

pd.DataFrame(leakage_checks).sort_values('roc_auc_single_feature', ascending=False)

## 4. Target, Status, and Out-Of-Time Split

In [ ]:
status_counts = df['status'].value_counts(dropna=False).rename_axis('status').reset_index(name='rows')
status_counts['pct'] = status_counts['rows'] / len(df)
display(status_counts)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=status_counts, x='status', y='rows', ax=ax, palette='Set2')
ax.set_title('Transaction Status Counts')
ax.set_xlabel('Status')
ax.set_ylabel('Rows')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2, p.get_height()), ha='center', va='bottom')
plt.show()

In [ ]:
SPLIT_DATE = pd.Timestamp('2025-10-01')

train = resolved[resolved['invoice_date'] < SPLIT_DATE].copy()
valid = resolved[resolved['invoice_date'] >= SPLIT_DATE].copy()
open_rows = df[df['defaulted'].isna()].copy()

split_summary = pd.DataFrame([
    {'split': 'train', 'rows': len(train), 'default_rate': train['defaulted'].mean(), 'start': train['invoice_date'].min(), 'end': train['invoice_date'].max()},
    {'split': 'validation', 'rows': len(valid), 'default_rate': valid['defaulted'].mean(), 'start': valid['invoice_date'].min(), 'end': valid['invoice_date'].max()},
    {'split': 'open_holdout', 'rows': len(open_rows), 'default_rate': np.nan, 'start': open_rows['invoice_date'].min(), 'end': open_rows['invoice_date'].max()},
])
split_summary

In [ ]:
monthly = (
    resolved.assign(invoice_month=resolved['invoice_date'].dt.to_period('M').dt.to_timestamp())
    .groupby('invoice_month')
    .agg(rows=('transaction_id', 'size'), defaults=('defaulted', 'sum'), default_rate=('defaulted', 'mean'))
    .reset_index()
)

fig, ax1 = plt.subplots(figsize=(13, 5))
sns.barplot(data=monthly, x='invoice_month', y='rows', color='#8fb3ff', ax=ax1)
ax1.set_title('Resolved Volume and Default Rate by Invoice Month')
ax1.set_xlabel('Invoice month')
ax1.set_ylabel('Resolved rows')
ax1.tick_params(axis='x', rotation=75)

ax2 = ax1.twinx()
ax2.plot(range(len(monthly)), monthly['default_rate'], color='#b22222', marker='o', linewidth=2)
ax2.axvline(monthly.index[monthly['invoice_month'].ge(SPLIT_DATE)].min() - 0.5, color='black', linestyle='--', linewidth=1)
ax2.set_ylabel('Default rate')
ax2.set_ylim(0, max(0.15, monthly['default_rate'].max() * 1.25))
plt.show()

## 5. Categorical Distributions

In [ ]:
categorical_to_plot = [
    'invoice_currency',
    'goods_category',
    'exp_segment',
    'exp_auditor_tier',
    'buy_creditsafe_rating',
    'buy_buyer_country',
    'exp_industry',
    'buy_buyer_sector',
]

def plot_top_counts(data, col, top_n=12, ax=None):
    counts = data[col].value_counts(dropna=False).head(top_n).reset_index()
    counts.columns = [col, 'rows']
    sns.barplot(data=counts, y=col, x='rows', ax=ax, color='#4c78a8')
    ax.set_title(f'Top {min(top_n, len(counts))}: {col}')
    ax.set_xlabel('Rows')
    ax.set_ylabel('')

fig, axes = plt.subplots(4, 2, figsize=(16, 20))
for ax, col in zip(axes.flat, categorical_to_plot):
    plot_top_counts(df, col, ax=ax)
plt.tight_layout()
plt.show()

## 6. Numeric Distributions

In [ ]:
numeric_summary = df[numeric_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
numeric_summary

In [ ]:
numeric_to_plot = [
    'invoice_amount_usd',
    'payment_terms_days',
    'advance_pct',
    'cumulative_outstanding_usd',
    'recent_velocity_count',
    'exp_annual_revenue_egp_millions',
    'exp_iscore',
    'exp_current_ratio',
    'exp_quick_ratio',
    'exp_debt_to_equity',
    'exp_operating_margin_pct',
    'exp_dscr',
    'exp_sales_growth_pct',
    'exp_fx_loss_to_npat_pct',
    'exp_historical_on_time_payment_rate',
    'exp_historical_dispute_rate',
    'exp_concentration_top1_buyer_pct',
    'buy_buyer_country_risk_score',
    'buy_buyer_payment_history_dso_avg',
]

fig, axes = plt.subplots(5, 4, figsize=(18, 20))
for ax, col in zip(axes.flat, numeric_to_plot):
    sns.histplot(df[col], kde=True, bins=35, ax=ax, color='#4c78a8')
    ax.set_title(col)
    ax.set_xlabel('')
for ax in axes.flat[len(numeric_to_plot):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 7. Default Rate by Segment

These views should be read as exploratory only. A high segment default rate can reflect small sample size, time mix, or confounding with other variables.

In [ ]:
def default_rate_table(data, col, min_rows=25):
    out = (
        data.groupby(col, dropna=False)
        .agg(rows=('transaction_id', 'size'), defaults=('defaulted', 'sum'), default_rate=('defaulted', 'mean'))
        .reset_index()
        .query('rows >= @min_rows')
        .sort_values(['default_rate', 'rows'], ascending=[False, False])
    )
    return out

segment_cols = [
    'goods_category',
    'exp_industry',
    'exp_segment',
    'exp_auditor_tier',
    'buy_buyer_country',
    'buy_creditsafe_rating',
    'insurance_covered',
    'docs_complete',
    'exp_audit_qualified',
]

for col in segment_cols:
    print(f'\n{col}')
    display(default_rate_table(resolved, col).head(15))

In [ ]:
plot_cols = ['goods_category', 'exp_auditor_tier', 'buy_buyer_country', 'buy_creditsafe_rating']

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
for ax, col in zip(axes.flat, plot_cols):
    tmp = default_rate_table(resolved, col).head(12).sort_values('default_rate')
    sns.barplot(data=tmp, y=col, x='default_rate', ax=ax, color='#b22222')
    ax.axvline(resolved['defaulted'].mean(), color='black', linestyle='--', linewidth=1)
    ax.set_title(f'Default rate by {col}')
    ax.set_xlabel('Default rate')
    ax.set_ylabel('')
plt.tight_layout()
plt.show()

## 8. First-Pass Cross-Sector Features

These are exploratory features that can apply across sectors. They are not final model features yet.

In [ ]:
EGP_PER_USD_ASSUMPTION = 50

eda = df.copy()
annual_revenue_usd_est = (eda['exp_annual_revenue_egp_millions'] * 1_000_000) / EGP_PER_USD_ASSUMPTION

eda['ticket_to_revenue_est'] = eda['invoice_amount_usd'] / annual_revenue_usd_est.replace(0, np.nan)
eda['outstanding_to_revenue_est'] = eda['cumulative_outstanding_usd'] / annual_revenue_usd_est.replace(0, np.nan)
eda['advance_tenor_exposure'] = eda['advance_pct'] * eda['payment_terms_days']
eda['liquidity_gap'] = eda['exp_current_ratio'] - eda['exp_quick_ratio']
eda['velocity_to_history'] = eda['recent_velocity_count'] / (eda['exp_historical_invoices_count'] + 1)

engineered_cols = [
    'ticket_to_revenue_est',
    'outstanding_to_revenue_est',
    'advance_tenor_exposure',
    'liquidity_gap',
    'velocity_to_history',
]

eda[engineered_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T

In [ ]:
eda_resolved = eda[eda['defaulted'].notna()].copy()
eda_resolved['defaulted'] = eda_resolved['defaulted'].astype(int)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, col in zip(axes.flat, engineered_cols):
    sns.boxplot(data=eda_resolved, x='defaulted', y=col, ax=ax, showfliers=False, palette='Set2')
    ax.set_title(f'{col} by default label')
    ax.set_xlabel('Defaulted')
for ax in axes.flat[len(engineered_cols):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 9. Numeric Correlation With Default

This excludes obvious leakage and outcome columns. Correlation is not causality, but it helps prioritize model and rule diagnostics.

In [ ]:
safe_numeric_cols = [
    c for c in eda_resolved.select_dtypes(include='number').columns
    if c not in LEAKY_OR_OUTCOME_COLS and c != 'defaulted'
]

corr_with_default = (
    eda_resolved[safe_numeric_cols + ['defaulted']]
    .corr(numeric_only=True)['defaulted']
    .drop('defaulted')
    .sort_values(key=lambda s: s.abs(), ascending=False)
    .to_frame('corr_with_default')
)
display(corr_with_default.head(25))

fig, ax = plt.subplots(figsize=(9, 8))
top_corr = corr_with_default.head(20).sort_values('corr_with_default')
sns.barplot(data=top_corr.reset_index(), y='index', x='corr_with_default', ax=ax, palette='coolwarm')
ax.axvline(0, color='black', linewidth=1)
ax.set_title('Top Numeric Correlations With Default')
ax.set_xlabel('Pearson correlation')
ax.set_ylabel('')
plt.show()

## 10. Peer-Group Outlier Exploration

The case asks for outlier detection within a sector peer group, not only in the whole dataset. This section computes simple within-`goods_category` z-scores as an exploratory baseline.

In [ ]:
peer_cols = [
    'invoice_amount_usd',
    'payment_terms_days',
    'advance_pct',
    'cumulative_outstanding_usd',
    'recent_velocity_count',
    'exp_debt_to_equity',
    'exp_dscr',
    'exp_historical_dispute_rate',
    'exp_concentration_top1_buyer_pct',
    'buy_buyer_payment_history_dso_avg',
    'ticket_to_revenue_est',
    'outstanding_to_revenue_est',
]

peer = eda.copy()
for col in peer_cols:
    peer_mean = peer.groupby('goods_category')[col].transform('mean')
    peer_std = peer.groupby('goods_category')[col].transform('std').replace(0, np.nan)
    peer[f'{col}_sector_z'] = (peer[col] - peer_mean) / peer_std

z_cols = [f'{c}_sector_z' for c in peer_cols]
peer['sector_outlier_feature_count'] = peer[z_cols].abs().gt(2.5).sum(axis=1)

peer_resolved = peer[peer['defaulted'].notna()].copy()
peer_resolved['defaulted'] = peer_resolved['defaulted'].astype(int)

outlier_summary = (
    peer_resolved.groupby('sector_outlier_feature_count')
    .agg(rows=('transaction_id', 'size'), defaults=('defaulted', 'sum'), default_rate=('defaulted', 'mean'))
    .reset_index()
)
outlier_summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=outlier_summary, x='sector_outlier_feature_count', y='default_rate', ax=ax, color='#b22222')
ax.axhline(peer_resolved['defaulted'].mean(), color='black', linestyle='--', linewidth=1)
ax.set_title('Default Rate by Count of Within-Sector Outlier Features')
ax.set_xlabel('Number of peer-group outlier features')
ax.set_ylabel('Default rate')
plt.show()

display(
    peer_resolved.sort_values('sector_outlier_feature_count', ascending=False)[
        ['transaction_id', 'goods_category', 'defaulted', 'sector_outlier_feature_count'] + peer_cols
    ].head(15)
)

## 11. Candidate Hard-Filter Diagnostics

This is exploratory evidence for Stage A. These are not final rules yet. A final hard filter must be justified by business policy, model evidence, and credit committee appetite.

In [ ]:
candidate_filters = {
    'missing_docs': ~resolved['docs_complete'],
    'iscore_below_550': resolved['exp_iscore'] < 550,
    'iscore_below_600': resolved['exp_iscore'] < 600,
    'non_cbe_auditor': resolved['exp_auditor_tier'].eq('Non-CBE'),
    'qualified_audit': resolved['exp_audit_qualified'],
    'buyer_country_risk_8_plus': resolved['buy_buyer_country_risk_score'] >= 8,
    'creditsafe_D_or_E': resolved['buy_creditsafe_rating'].isin(['D', 'E']),
    'no_insurance': ~resolved['insurance_covered'],
    'high_dispute_rate_5pct_plus': resolved['exp_historical_dispute_rate'] >= 0.05,
    'low_dscr_below_1': resolved['exp_dscr'] < 1.0,
    'high_leverage_above_3': resolved['exp_debt_to_equity'] > 3.0,
}

filter_rows = []
overall_default_rate = resolved['defaulted'].mean()

for name, mask in candidate_filters.items():
    subset = resolved[mask]
    filter_rows.append({
        'filter': name,
        'hit_rows': int(mask.sum()),
        'hit_rate': mask.mean(),
        'default_rate_if_hit': subset['defaulted'].mean() if len(subset) else np.nan,
        'lift_vs_overall': (subset['defaulted'].mean() / overall_default_rate) if len(subset) else np.nan,
        'defaults_captured': int(subset['defaulted'].sum()) if len(subset) else 0,
        'pct_all_defaults_captured': subset['defaulted'].sum() / resolved['defaulted'].sum() if len(subset) else 0,
    })

filter_diagnostics = pd.DataFrame(filter_rows).sort_values('lift_vs_overall', ascending=False)
filter_diagnostics

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
tmp = filter_diagnostics.sort_values('default_rate_if_hit')
sns.barplot(data=tmp, y='filter', x='default_rate_if_hit', ax=ax, color='#b22222')
ax.axvline(overall_default_rate, color='black', linestyle='--', linewidth=1, label='Overall default rate')
ax.set_title('Candidate Hard Filter Default Rates')
ax.set_xlabel('Default rate among rows hit by filter')
ax.set_ylabel('')
ax.legend()
plt.show()

## 12. Exporter and Buyer Concentration

In [ ]:
def entity_summary(data, entity_col, min_rows=5):
    return (
        data.groupby(entity_col)
        .agg(rows=('transaction_id', 'size'), defaults=('defaulted', 'sum'), default_rate=('defaulted', 'mean'), total_invoice_usd=('invoice_amount_usd', 'sum'))
        .query('rows >= @min_rows')
        .sort_values(['default_rate', 'rows'], ascending=[False, False])
    )

exporter_risk = entity_summary(resolved, 'exporter_id', min_rows=8)
buyer_risk = entity_summary(resolved, 'buyer_id', min_rows=5)

print('Exporters with at least 8 resolved deals, sorted by default rate')
display(exporter_risk.head(15))

print('Buyers with at least 5 resolved deals, sorted by default rate')
display(buyer_risk.head(15))

## 13. Open Holdout Sanity Checks

Open rows have no usable label and should not be included in validation. We can still score them later and check whether predicted probabilities look plausible for a newer book.

In [ ]:
open_summary = pd.DataFrame({
    'resolved_mean': resolved[numeric_to_plot].mean(),
    'open_mean': open_rows[numeric_to_plot].mean(),
})
open_summary['relative_diff_open_vs_resolved'] = (open_summary['open_mean'] - open_summary['resolved_mean']) / open_summary['resolved_mean'].replace(0, np.nan)
open_summary.sort_values('relative_diff_open_vs_resolved', key=lambda s: s.abs(), ascending=False).head(20)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_top_counts(open_rows, 'goods_category', ax=axes[0])
plot_top_counts(open_rows, 'buy_buyer_country', ax=axes[1])
axes[0].set_title('Open Rows by Goods Category')
axes[1].set_title('Open Rows by Buyer Country')
plt.tight_layout()
plt.show()

## 14. Notes for the Next Build Step

Use this EDA to drive the first implementation pass:

- create a reusable data loader and split function
- encode the leakage contract in code so leaky columns cannot enter the model accidentally
- move candidate hard-filter thresholds into a config file
- formalize cross-sector engineered features
- replace this exploratory peer z-score with a reusable peer-group outlier component
- keep open rows separate for post-training sanity scoring